# Part 7 · Notebook 04 — Volatility overlays, the Hurst exponent and pairs

**Sessions:** S4 (Either-way, volatility, mathematical & statistical groups) · [Lesson plan](../../docs/lessons/PART_07_STRATEGY_LIBRARY.md) · graded labs in [`labs/part07/`](../../labs/part07/)

**You will:**
1. Put a volatility-target overlay on any strategy.
2. Measure trending vs mean-reverting behaviour with the Hurst exponent.
3. Build a pairs trade whose hedge ratio never sees the future.
4. Watch a pair break.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic, built from regimes you know, and every strategy here is a **hypothesis** with a first-look evaluation: the honest backtest comes in Part 8.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p7lib.py is in notebooks/part07/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p7lib as p

p.use_course_style()

In [ ]:
bars = p.regime_market()
o, c = bars.open.to_numpy(), bars.close.to_numpy()

## 1. The volatility-target overlay

Scale any position so its risk is roughly constant: multiply by `target / realized vol`, where realized vol is the annualized rolling standard deviation (`ddof=1`) of the returns **known at each close** over the last `n` bars. Clip the scale to `[0, max_lev]`, and use 0 where the vol is not known yet (`np.nan_to_num`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def vol_target_overlay(pos, returns, target=0.10, n=20, max_lev=2.0):
    vol = pd.Series(returns).rolling(n).std().to_numpy() * np.sqrt(252)
    scale = ...                                   # ✍️ target / vol, 0 where vol is 0, clipped to [0, max_lev]
    return np.asarray(pos, dtype=float) * np.nan_to_num(scale)

ret = np.r_[0.0, c[1:] / c[:-1] - 1]              # close-to-close return known at each close
mine = p.attempt(vol_target_overlay, np.ones(len(c)), ret)
mine = p.check("vol_target_overlay", mine, p.vol_target_overlay(np.ones(len(c)), ret))
for name, pos in [("buy & hold", np.ones(len(c))), ("buy & hold, vol-targeted to 10%", mine)]:
    s = p.summary(p.quick_eval(pos, o))
    print(f"{name:32s} Sharpe {s['sharpe']:+.2f}  max drawdown {s['max_dd']:.0%}")

## 2. The Hurst exponent

How does the spread of price changes grow with the horizon? For a random walk, `std(x[t+lag] − x[t]) ∝ lag^0.5`. Faster growth (`H > 0.5`) means moves persist (trending); slower (`H < 0.5`) means they undo themselves (mean reverting). Estimate `H` as the slope of `log std` on `log lag` (`np.polyfit(..., 1)[0]`). Pass **log** prices.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def hurst(x, lags=range(2, 64)):
    lags = np.asarray(list(lags))
    tau = ...                                     # ✍️ the std of x[lag:] − x[:-lag] for each lag
    return float(np.polyfit(np.log(lags), np.log(tau), 1)[0])

logc = np.log(c)
blocks = {f"block {i // 125:2d} ({'trend' if bars.trend.iloc[i] else 'range'})": logc[i:i + 125] for i in range(0, 750, 125)}
mine = {k: p.attempt(hurst, v) for k, v in blocks.items()}
mine = p.check("hurst", mine, {k: p.hurst_exponent(v) for k, v in blocks.items()})
pd.Series(mine).round(2)

Single blocks are noisy. Average over all 24 blocks by their **true** regime (with lags up to 19 bars, short enough for 125-bar blocks), and compare with a pure random walk:

In [ ]:
by_regime = {"range-bound": [], "trending": []}
for i in range(0, len(logc), 125):
    by_regime["trending" if bars.trend.iloc[i] else "range-bound"].append(p.hurst_exponent(logc[i:i + 125], range(2, 20)))
rw = np.cumsum(np.random.default_rng(0).normal(0, 0.01, 3000))
display(pd.DataFrame({k: {"mean H": np.mean(v), "sd across blocks": np.std(v), "blocks": len(v)} for k, v in by_regime.items()}).T.round(2))
print(f"random walk: H = {p.hurst_exponent(rw, range(2, 20)):.2f}")

Range-bound blocks sit clearly below 0.5 on average. The trending blocks sit near 0.5, not above it: their trend is a steady **drift** plus independent noise, and the Hurst exponent measures whether *changes* persist, which a drift doesn't create (the standard deviation removes it). And block-to-block scatter is large. A regime label estimated from the data is a noisy guess at the truth; the lab's Clinic W1 compares strategies under true and estimated labels.

## 3. A pairs trade without look-ahead

Two log prices with `y ≈ β·x + spread`, where the spread mean-reverts. At bar `t`, estimate `β` by regression on the **previous** `lookback` bars only, then compare today's spread with that window. Write the window: it must end at `t − 1`.

In [ ]:
y, x = p.cointegrated_pair()
plt.figure(figsize=(10, 3.2)); plt.plot(y, label="log y"); plt.plot(x, label="log x"); plt.legend(); plt.title("A cointegrated pair (true β = 1.4)"); plt.show()

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def pairs(y, x, lookback=60, entry=2.0, exit_=0.5):
    pos, betas = np.zeros(y.size), np.full(y.size, np.nan)
    for t in range(lookback, y.size):
        ys, xs = ...                              # ✍️ the previous `lookback` bars of y and x (t itself excluded)
        beta = np.polyfit(xs, ys, 1)[0]
        betas[t] = beta
        spread = ys - beta * xs
        sd = spread.std()
        z = (y[t] - beta * x[t] - spread.mean()) / sd if sd > 0 else 0.0
        prev = pos[t - 1]
        if prev == 0:
            pos[t] = 1.0 if z < -entry else (-1.0 if z > entry else 0.0)
        else:
            pos[t] = 0.0 if abs(z) < exit_ else prev
    return pos, betas

res = p.attempt(pairs, y, x)
mine = list(res) if res is not Ellipsis else Ellipsis
mine = p.check("pairs", mine, list(p.pairs_positions(y, x)))
pnl = p.pairs_pnl(mine[0], y, x, mine[1])
print(f"spread trade: Sharpe {pnl.mean() / pnl.std() * np.sqrt(252):.2f}, mean estimated β {np.nanmean(mine[1]):.2f}")

## 4. When the pair breaks

Same pair, but at bar 900 the relationship changes (β drops by 40%). The rolling hedge ratio adapts, slowly, and the spread "reverts" to a level that no longer exists. That is why pairs books need a break test and a retirement rule (Part 9).

In [ ]:
y2, x2 = p.cointegrated_pair(break_at=900)
pos2, b2 = p.pairs_positions(y2, x2)
pnl2 = p.pairs_pnl(pos2, y2, x2, b2)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(b2); axes[0].axvline(900, color=p.PALETTE[7], ls="--"); axes[0].set_title("Rolling β (true 1.4, then 0.84)")
axes[1].plot(np.cumsum(pnl2)); axes[1].axvline(900, color=p.PALETTE[7], ls="--"); axes[1].set_title("Cumulative spread P&L (log units)")
plt.tight_layout(); plt.show()
print(f"P&L before the break {pnl2[:900].sum():+.3f}, after {pnl2[900:].sum():+.3f}")

## Wrap-up

* Volatility targeting is an overlay for any strategy: it evens out risk, and here it cuts the drawdown.
* Hurst and similar statistics label regimes, noisily.
* Every estimated input (β, mean, std) uses only past bars; relationships break, so monitor them.
* Graded version: `labs/part07/week23_linear_groups` (VIX-regime overlay, Kalman level, turn-of-month, overnight vs intraday too).